<div dir="rtl">

# الدرس 49: ابنِ نموذج جي بي تي صغيراً ودرّبه

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### البيانات

</div>

In [ ]:
import math
import os
import time
import urllib.request
import torch

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(URL, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", encoding="utf-8").read()

chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

T, B = 64, 32                                    # context window, batch size

def get_batch(d, generator):
    starts = torch.randint(len(d) - T - 1, (B,), generator=generator)
    x = torch.stack([d[i:i + T] for i in starts])
    y = torch.stack([d[i + 1:i + T + 1] for i in starts])     # shifted by one
    return x, y

x, y = get_batch(train_data, torch.Generator().manual_seed(0))
print("batch shapes:", tuple(x.shape), tuple(y.shape))
print("input :", repr(decode(x[0, :20].tolist())))
print("target:", repr(decode(y[0, :20].tolist())))

<div dir="rtl">

### النموذج

</div>

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn.functional as F
from torch import nn

@dataclass
class GPTConfig:
    vocab_size: int = 50257
    block_size: int = 1024          # context window
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    dropout: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = cfg.dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, 4 * cfg.d_model)
        self.proj = nn.Linear(4 * cfg.d_model, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))      # tokens communicate
        x = x + self.mlp(self.ln2(x))       # each token computes
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):                            # GPT-2 style initialization
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, "sequence longer than the context window"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                   # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
torch.manual_seed(0)
cfg = GPTConfig(vocab_size=len(chars), block_size=T, n_layer=4, n_head=4, d_model=128, dropout=0.0)
model = GPT(cfg)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

<div dir="rtl">

### التوليد: قبل التدريب

</div>

In [ ]:
@torch.no_grad()
def generate(model, prompt, n_new, temperature=1.0, seed=0):
    model.eval()
    g = torch.Generator().manual_seed(seed)
    idx = torch.tensor([encode(prompt)])
    for _ in range(n_new):
        logits, _ = model(idx[:, -T:])                          # crop to the context window
        probs = torch.softmax(logits[0, -1] / temperature, dim=-1)   # last position only
        nxt = torch.multinomial(probs, 1, generator=g)
        idx = torch.cat([idx, nxt[None]], dim=1)
    model.train()
    return decode(idx[0].tolist())

print(generate(model, "ROMEO:\n", 150))

<div dir="rtl">

### التدريب

</div>

In [ ]:
steps, warmup = 3000, 100
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

def lr_factor(step):                               # warmup, then cosine decay
    return min(1.0, (step + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(step, steps) / steps))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)

@torch.no_grad()
def evaluate(n_batches=20):
    model.eval()
    result = []
    for d in (train_data, val_data):
        g = torch.Generator().manual_seed(1)        # the same batches every time
        losses = [model(*get_batch(d, g))[1].item() for _ in range(n_batches)]
        result.append(sum(losses) / n_batches)
    model.train()
    return result

g = torch.Generator().manual_seed(0)
history, t0 = [], time.time()
for step in range(steps + 1):
    if step % 500 == 0:
        tr_loss, va_loss = evaluate()
        history.append((step, tr_loss, va_loss))
        print(f"step {step:4}: train {tr_loss:.3f} | val {va_loss:.3f} | {time.time() - t0:4.0f}s")
    if step == steps:
        break
    x, y = get_batch(train_data, g)
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()

In [ ]:
import matplotlib.pyplot as plt
s, tr, va = zip(*history)
plt.figure(figsize=(7, 3.6))
plt.plot(s, tr, "o-", label="train"); plt.plot(s, va, "o-", label="validation")
plt.axhline(math.log(65), ls=":", color="gray"); plt.text(3000, math.log(65) + 0.05, "ln 65: random", ha="right", fontsize=9)
plt.axhline(2.48, ls="--", color="gray"); plt.text(3000, 2.53, "bigram model (lesson 45)", ha="right", fontsize=9)
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(loc="center right"); plt.grid(True, alpha=0.4)
plt.show()

<div dir="rtl">

### التوليد: بعد التدريب

</div>

In [ ]:
print(generate(model, "ROMEO:\n", 400, temperature=0.8))

<div dir="rtl">

### الحفظ

</div>

In [ ]:
torch.save({"model": model.state_dict(), "config": cfg.__dict__, "chars": chars}, "tiny_gpt.pt")
print("saved tiny_gpt.pt,", os.path.getsize("tiny_gpt.pt") // 1024, "KB")

<div dir="rtl">

### ماذا تنظر إليه الرؤوس؟

</div>

In [ ]:
@torch.no_grad()
def attention_maps(model, s, layer):
    model.eval()
    idx = torch.tensor([encode(s)])
    x = model.tok_emb(idx) + model.pos_emb(torch.arange(idx.shape[1]))
    for block in model.blocks[:layer]:
        x = block(x)
    blk = model.blocks[layer]
    q, k, _ = blk.attn.qkv(blk.ln1(x)).split(cfg.d_model, dim=2)
    n, h = idx.shape[1], cfg.n_head
    q = q.view(1, n, h, -1).transpose(1, 2)
    k = k.view(1, n, h, -1).transpose(1, 2)
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    scores = scores.masked_fill(torch.triu(torch.ones(n, n, dtype=torch.bool), 1), float("-inf"))
    model.train()
    return scores.softmax(-1)[0]                   # (heads, n, n)

sample = "KING RICHARD:\nMy lord"
maps = attention_maps(model, sample, layer=0)
for head in range(cfg.n_head):
    focus = maps[head].argmax(-1)                  # which position each character looks at most
    print(f"head {head}: most attended offsets", (focus - torch.arange(len(sample))).tolist()[-8:])

In [ ]:
fig, axes = plt.subplots(1, cfg.n_head, figsize=(12, 3.4))
labels = [c if c != "\n" else "⏎" for c in sample]
for head, ax in enumerate(axes):
    ax.imshow(maps[head], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"layer 0, head {head}", fontsize=9)
    ax.set_xticks(range(len(sample))); ax.set_xticklabels(labels, fontsize=6)
    ax.set_yticks(range(len(sample))); ax.set_yticklabels(labels, fontsize=6)
plt.tight_layout()
plt.show()

<div dir="rtl">

## كبّر التجربة على كرت الشاشة

</div>

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
T, B = 256, 64                                          # longer context, bigger batch
cfg = GPTConfig(vocab_size=len(chars), block_size=T, n_layer=6, n_head=6, d_model=384, dropout=0.2)
model = GPT(cfg).to(device)
# inside the training loop, move every batch to the same device:
#     x, y = x.to(device), y.to(device)
# and train for about 5000 steps

<div dir="rtl">

## تمرين: جرّب البدايات والحرارة

1. ولّد 200 حرف من ثلاث بدايات مختلفة: اسم شخصية من المسرحيات، وسطر فارغ، وجملة من اختيارك.
2. ولّد من نفس البداية بحرارة 0.5، ثم 1.0، ثم 1.5. صف الفرق.
3. لماذا لا نحتاج أن نعيد التدريب لكي نغيّر الحرارة؟

</div>

In [ ]:
# اكتب حلّك هنا
